In [ ]:
import pandas as pd
import re

email_data = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Email Qwen Priority Dataset/priority_synthetic_balanced_full_dataset.csv')
#email_data = pd.read_csv('C:/Users/User/OneDrive - Asia Pacific University/APU Final Year Project/FYP Email Project Documents/Email Dataset Python/Email Qwen Priority Dataset/realistic_priority_email_dataset_full_format.csv')

In [ ]:
email_data.info(memory_usage='deep') 
print("Number of Rows and Columns: ", email_data.shape) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108296 entries, 0 to 108295
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   From             108296 non-null  object
 1   To               108296 non-null  object
 2   Subject          108296 non-null  object
 3   Message          108296 non-null  object
 4   Day              108296 non-null  object
 5   Date & Time      108296 non-null  object
 6   url_count        108296 non-null  int64 
 7   Cleaned_Subject  108296 non-null  object
 8   Cleaned_Message  108296 non-null  object
 9   Combined_Text    108296 non-null  object
 10  Subject_LLM      108296 non-null  object
 11  Message_LLM      108296 non-null  object
 12  Email_Priority   108296 non-null  object
dtypes: int64(1), object(12)
memory usage: 176.0 MB
Number of Rows and Columns:  (108296, 13)


In [ ]:
email_data

In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])  # Only tokenizer + lemmatizer

def advanced_cleaning_pipeline(email_text):
    if not isinstance(email_text, str):
        return ""

    # 1. Remove forwarded/original headers (multi-line removal)
    email_text = re.sub(r"(?is)-----.*?Subject:", "", email_text)
    
    # 1.5 Remove "Dear [Name],"
    email_text = re.sub(r"(?i)^dear\s+\[name\],?\s*", "", email_text)

    # 1.6 Remove "Best regards," and [Sender]
    email_text = re.sub(r"(?i)^best regards,?\s*\[sender\]\s*$", "", email_text)
    email_text = re.sub(r"\[name\]", "", email_text, flags=re.IGNORECASE)
    email_text = re.sub(r"\[sender\]", "", email_text, flags=re.IGNORECASE)
    
    # 1.7 Remove specific greetings like "Hi Team,", "Team,", "Hello,", "Hi,"
    email_text = re.sub(r"(?im)^(hi team,|team,|hello,|hi,)\s*", "", email_text)

    # 2. Normalize all newlines, tabs, excessive spaces
    email_text = re.sub(r"[\r\n\t]+", " ", email_text)
    email_text = re.sub(r"\s+", " ", email_text).strip()

    # 3. Fix common email encoding artifacts
    email_text = re.sub(r"=20", " ", email_text)  # Quoted printable space
    email_text = re.sub(r"=09", " ", email_text)  # Quoted printable tab
    email_text = re.sub(r"=3D", "=", email_text)  # Quoted printable equals
    email_text = re.sub(r"=\s?", "", email_text)  # Quoted printable soft line break

    # 4. Remove URLs, Emails, Attachments (no placeholders - full removal)
    email_text = re.sub(r"http[s]?://\S+", "", email_text)
    email_text = re.sub(r"\b\S+@\S+\b", "", email_text)
    email_text = re.sub(r"<< File:.*?>>", "", email_text)

    # 5. Remove common signatures/closings
    email_text = re.sub(r"(?i)(thanks|regards|sincerely|best),?", "", email_text)

    # 6. Optional: Remove phone numbers
    email_text = re.sub(r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}", "", email_text)

    # 7. Final cleanup - extra spaces from removals
    email_text = re.sub(r"\s+", " ", email_text).strip()

    # 8. Run spaCy NLP (tokenization, lemmatization, stopword removal)
    doc = nlp(email_text)

    # 9. Keep only lemmatized tokens (no stopwords, no punctuation, no numbers)
    cleaned_tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.like_num
    ]

    cleaned_text = " ".join(cleaned_tokens)

    # 10. Optional: Truncate to max 300 words (recommended for classification)
    max_length = 300
    if len(cleaned_text.split()) > max_length:
        cleaned_text = " ".join(cleaned_text.split()[:max_length])

    return cleaned_text

email_data['Cleaned_Message'] = email_data['Message'].apply(advanced_cleaning_pipeline)

In [ ]:
email_data

In [ ]:
def universal_artifact_cleaner(text):
    if not isinstance(text, str):
        return ""

    # Common encoding artifacts (Quoted Printable)
    text = re.sub(r'=\s?', '', text)   # Soft line breaks
    text = re.sub(r'=20', ' ', text)   # Space encoding
    text = re.sub(r'=09', ' ', text)   # Tab encoding
    text = re.sub(r'=3D', '=', text)   # Equals sign encoding

    # Weird numerical artifacts (like 01,s for 's)
    text = re.sub(r'(\d{2})[,\.](\w)', r'\2', text)  # Remove numeric prefix
    text = re.sub(r'\b(\d{2})\b', '', text)          # Remove stray numbers if isolated (optional)

    # Fix apostrophe handling
    text = re.sub(r'01,s', "'s", text)
    text = re.sub(r'01,t', "'t", text)

    # Common email artifacts (forwarded/reply headers)
    text = re.sub(r'(?is)-----.*?Subject:', '', text)  # Forwarded/Original block

    # Remove weird standalone letters (like v)
    text = re.sub(r'\bv\b', '', text)

    # Extra space cleanup
    text = re.sub(r'\s+', ' ', text).strip()

    return text

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(universal_artifact_cleaner)

In [ ]:
import re

def further_process_flat(text):
    if not isinstance(text, str):
        return ""

    # Remove repeated periods or replace with space (you want no noise from this)
    text = re.sub(r'\s*\.\s*\.\s*', ' ', text)

    # Optional: Remove ALL periods if you want zero punctuation
    text = re.sub(r'\.', ' ', text)

    # Normalize spacing
    text = re.sub(r'\s+', ' ', text).strip()

    # Lowercase everything (best practice for TF-IDF)
    text = text.lower()

    return text

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(further_process_flat)

In [ ]:
import re

def clean_promo_email_symbol_free(text):
    if not isinstance(text, str):
        return ""

    # Fix encoding artifacts and remove problematic symbols
    text = text.replace("men?s", "mens")
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)  # Remove all symbols

    # Normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Lowercase everything for consistent TF-IDF input
    return text.lower()

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(clean_promo_email_symbol_free)

In [ ]:
import re

def remove_urls(email_text):
    if not isinstance(email_text, str):
        return ""  # Return empty string if the input is not a string
    
    # Remove URL-like patterns (e.g., "http www expedia com", "http://www.example.com", "www.example.com")
    email_text = re.sub(r'\b(?:http|https|www)\S*\b', '', email_text)

    # Remove excessive spaces
    email_text = re.sub(r'\s+', ' ', email_text).strip()

    return email_text

# Apply to your DataFrame
email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(remove_urls)


In [ ]:
#token and stop words
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    
    if not text or not isinstance(text, str):
        return []
    
    text = text.lower()
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    return filtered_tokens 

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(tokenize_text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
import re

def smart_untokenizer(tokens):
    text = ' '.join(tokens)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)  # remove space before punctuation
    text = re.sub(r"(\()\s+", r"\1", text)        # fix spacing after (
    text = re.sub(r"\s+(\))", r"\1", text)        # fix spacing before )
    return text.strip()

email_data['Cleaned_Message'] = email_data['Cleaned_Message'].apply(smart_untokenizer)


In [ ]:
email_data

# Subject Cleaning

In [ ]:
email_data.info()

In [ ]:
import re

def clean_subject_line(subject):
    if not isinstance(subject, str):
        return ""

    # 1. Remove common prefixes like "FW:", "Fwd:", "RE:", "[RE]", etc.
    subject = re.sub(r'^\[(FW|FWD|RE|REPLIED|FORWARDED)\]\s*', '', subject)  # Handling brackets around prefixes
    subject = re.sub(r'^(FW:|Fwd:|RE:|Re:|Replied:|Sent:|Reply:|Fwd\[1\]:|Re\[1\]:)\s*', '', subject)

    # 2. Remove excessive spaces
    subject = re.sub(r'\s+', ' ', subject).strip()

    return subject

# Apply to the DataFrame's Subject column
email_data['Cleaned_Subject'] = email_data['Subject'].apply(clean_subject_line)

In [ ]:
import re

# Function to remove standalone numbers (like '11') but keep words like 'Warning11'
def remove_standalone_numbers(text):
    text = re.sub(r'\b\d+\b', '', text)  # Remove standalone numbers
    return text.strip()

# Function to remove special characters/symbols (like punctuation)
def remove_special_characters(text):
    text = re.sub(r'[^\w\s]', '', text)  # Remove anything that's not alphanumeric or space
    return text

def combine_cleaning_steps(text):
    if not isinstance(text, str):
        # Handle non-string values gracefully
        text = str(text) if text is not None else ''
    
    # First, remove the standalone numbers
    text = remove_standalone_numbers(text)
    
    # Then, remove special characters
    text = remove_special_characters(text)
    
    return text

# Apply the function to the subject column
email_data['Cleaned_Subject'] = email_data['Cleaned_Subject'].apply(combine_cleaning_steps)


In [ ]:
# Count NaN, None, and empty string values in the 'Cleaned_Subject' column
nan_count = email_data['Cleaned_Subject'].isna().sum() + (email_data['Cleaned_Subject'] == '').sum()

# Print the result
print(f"Number of NaN or empty values in 'Cleaned_Subject': {nan_count}")

Number of NaN or empty values in 'Cleaned_Subject': 0


In [ ]:
# Count NaN, None, and empty string values in the 'Cleaned_Subject' column
nan_count = email_data['Cleaned_Message'].isna().sum() + (email_data['Cleaned_Message'] == '').sum()

# Print the result
print(f"Number of NaN or empty values in 'Cleaned_Message': {nan_count}")

Number of NaN or empty values in 'Cleaned_Message': 0


In [ ]:
#token and stop words
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
stop_words = set(stopwords.words('english'))

def tokenize_text(text):
    
    if not text or not isinstance(text, str):
        return []
    
    text = text.lower()
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    return filtered_tokens 

email_data['Cleaned_Subject'] = email_data['Cleaned_Subject'].apply(tokenize_text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
import re

def smart_untokenizer(tokens):
    text = ' '.join(tokens)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)  # remove space before punctuation
    text = re.sub(r"(\()\s+", r"\1", text)        # fix spacing after (
    text = re.sub(r"\s+(\))", r"\1", text)        # fix spacing before )
    return text.strip()

email_data['Cleaned_Subject'] = email_data['Cleaned_Subject'].apply(smart_untokenizer)

In [ ]:
# Combine Subject and Message into Combined_Text
email_data['Combined_Text'] = email_data['Cleaned_Subject'] + " " + email_data['Cleaned_Message']

# Final step: Remove accidental 'nan' (in case subject or message was missing) and extra spaces
email_data['Combined_Text'] = email_data['Combined_Text'].str.replace(r'\b[nN][aA][nN]\b', '', regex=True).str.strip()

In [ ]:
# Count NaN, None, and empty string values in the 'Cleaned_Subject' column
nan_count = email_data['Combined_Text'].isna().sum() + (email_data['Combined_Text'] == '').sum()

# Print the result
print(f"Number of NaN or empty values in 'Combined_Text': {nan_count}")

Number of NaN or empty values in 'Combined_Text': 0


# Subject LLM Cleaning

In [ ]:
import pandas as pd
import re
from bs4 import BeautifulSoup

def clean_email_data(email_data):
    """
    Function to clean 'Subject' and 'Message' columns:
    - Strips leading/trailing spaces
    - Removes special characters and symbols
    - Removes HTML tags
    - Stores cleaned versions in new columns 'Subject_LLM' and 'Message_LLM'
    """
    
    def clean_text(text):
        """
        Helper function to clean the text by:
        - Stripping leading/trailing spaces
        - Removing special characters and symbols
        - Removing HTML tags
        - Handling non-string values
        """
        if not isinstance(text, str):  # Ensure the text is a string
            text = str(text) if text is not None else ""  # Convert non-string to an empty string if None

        # Step 1: Remove HTML tags
        text = BeautifulSoup(text, "html.parser").get_text()

        # Step 2: Remove special characters or symbols (allow letters, numbers, and spaces)
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

        # Step 3: Strip leading and trailing spaces
        text = text.strip()

        return text

    # Step 1: Clean 'Subject' and 'Message' columns and store the cleaned text in new columns
    email_data['Subject_LLM'] = email_data['Subject'].apply(clean_text)
    email_data['Message_LLM'] = email_data['Message'].apply(clean_text)

    return email_data

# Clean the email data
email_data_cleaned = clean_email_data(email_data)

In [ ]:
email_data_cleaned

In [ ]:
# Save the DataFrame to a CSV file
email_data_cleaned.to_csv('cleaned_priority_synthetic_balanced_full_dataset.csv', index=False)